In [1]:
#qwopus with updated prompt now, refinement, clean file

In [2]:
!nvidia-smi

Thu Apr  9 12:39:09 2026       
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 535.261.03             Driver Version: 535.261.03   CUDA Version: 12.2     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  NVIDIA H100 80GB HBM3          On  | 00000000:19:00.0 Off |                    0 |
| N/A   40C    P0              72W / 700W |      0MiB / 81559MiB |      0%      Default |
|                                         |                      |             Disabled |
+-----------------------------------------+----------------------+--

In [3]:
#old model 
'''
from llama_cpp import Llama

llm = Llama.from_pretrained(
    repo_id="Jackrong/Qwen3.5-27B-Claude-4.6-Opus-Reasoning-Distilled-GGUF",
    filename="*Q4_K_M.gguf",
    n_ctx=24576,
    n_gpu_layers=-1,
    verbose=False,
)
print("model loaded")
'''

#new model
from llama_cpp import Llama

llm = Llama.from_pretrained(
    repo_id="Jackrong/Qwopus3.5-27B-v3-GGUF",
    filename="*Q8_0.gguf",
    n_ctx=24576,
    n_gpu_layers=-1,
    verbose=False,
)
print("model loaded")

/home/mprakash/vllm_env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
llama_context: n_ctx_seq (24576) < n_ctx_train (262144) -- the full capacity of the model will not be utilized


model loaded


In [4]:
import json, time, os, csv, re
from datetime import datetime

DATA_PATH = "gold_standard_final_15.json"
os.makedirs("benchmark_results", exist_ok=True)

with open(DATA_PATH, "r") as f:
    papers = json.load(f)

In [6]:
## TRYING WIHT 1 call on QWOPUS v3 since 2 prompt approach not work

PROMPT_TEMPLATE = """Extract all microbe-disease relationships from this paper.

For each microbe with a reported change, record:
- taxon_name: exact name from the paper
- direction: "increased" or "decreased"

Read the Abstract, Results, and Discussion sections. Ignore Methods, figures, tables, and supplementary materials. Extract at every taxonomic level mentioned. Be forgiving on microbe names but strict on directionality — if direction is unclear, skip that microbe.

Return ONLY valid JSON:
{{"disease": "disease name", "taxa_enriched": ["taxon1", "taxon2"], "taxa_depleted": ["taxon3", "taxon4"]}}

If no clear microbe changes found, return:
{{"disease": "disease name", "taxa_enriched": [], "taxa_depleted": []}}

Paper text:
{text}

JSON:"""

def smart_truncate(text):
    for marker in ["References\n", "REFERENCES\n", "Bibliography\n"]:
        idx = text.rfind(marker)
        if idx > 0:
            return text[:idx]
    return text


results = []

for i, paper in enumerate(papers):
    print(f"[{i+1}/{len(papers)}] {paper['title'][:70]}...")
    text = smart_truncate(paper["text"])
    start = time.time()

    output = llm.create_chat_completion(
        messages=[{"role": "user", "content": PROMPT_TEMPLATE.format(text=text)}],
        temperature=0, max_tokens=2048,
    )
    raw = output["choices"][0]["message"]["content"]
    elapsed = time.time() - start

    try:
        parsed = parse_output(raw)
    except:
        parsed = {"disease": paper["disease"], "taxa_enriched": [], "taxa_depleted": [], "parse_error": True}

    results.append({
        "title": paper["title"],
        "disease": paper["disease"],
        "in_gold_standard": paper["in_gold_standard"],
        "expected_enriched": paper.get("taxa_enriched", ""),
        "expected_depleted": paper.get("taxa_depleted", ""),
        "predicted_enriched": ", ".join(parsed.get("taxa_enriched", [])),
        "predicted_depleted": ", ".join(parsed.get("taxa_depleted", [])),
        "predicted_disease": parsed.get("disease", ""),
        "time_seconds": round(elapsed, 2),
        "parse_error": parsed.get("parse_error", False),
    })

    print(f"  enriched: {parsed.get('taxa_enriched', [])}")
    print(f"  depleted: {parsed.get('taxa_depleted', [])}")
    print(f"  {elapsed:.1f}s")

print(f"\nDone. {len(results)} papers.")

[1/15] Intestinal flora induces depression by mediating the dysregulation of ...
  enriched: []
  depleted: []
  30.7s
[2/15] Gut microbes exacerbate systemic inflammation and behavior disorders i...
  enriched: []
  depleted: []
  27.4s
[3/15] Alterations in gut microbiota and metabolomic profiles in acute stroke...
  enriched: []
  depleted: []
  48.1s
[4/15] Gut microbiome dysbiosis across early Parkinson's disease, REM sleep b...
  enriched: []
  depleted: []
  49.8s
[5/15] The gut microbiota in multiple sclerosis varies with disease activity....
  enriched: []
  depleted: []
  50.1s
[6/15] Gut Microbial Ecosystem in Parkinson Disease: New Clinicobiological In...
  enriched: []
  depleted: []
  26.2s
[7/15] Dysbiosis of gut microbiota in a selected population of Parkinson's pa...
  enriched: []
  depleted: []
  11.6s
[8/15] Gut microbiota distinguishes aging hispanics with Alzheimer's disease:...
  enriched: []
  depleted: []
  48.2s
[9/15] Examining the complex Interplay between g

In [5]:
'''
PROMPT_CALL1 = """Extract all microbe-disease relationships from this paper.

Read the Abstract, Results, and Discussion sections. Ignore Methods, figures, tables, and supplementary materials. Be forgiving on microbe names but strict on directionality — if direction is unclear, skip that microbe.

Return ONLY valid JSON:
{{"disease": "disease name", "taxa_enriched": ["taxon1", "taxon2"], "taxa_depleted": ["taxon3", "taxon4"]}}

If no clear microbe changes found, return:
{{"disease": "disease name", "taxa_enriched": [], "taxa_depleted": []}}

Paper text:
{text}

JSON:"""

PROMPT_CALL2 = """You previously extracted these microbe-disease relationships from a paper:

{first_pass_json}

Now re-read the paper and check:
1. Did you miss any microbes mentioned in the Results or Discussion that had a clear direction?
2. Did you include any microbes where the direction was actually unclear?
3. Did you accidentally pull from Methods or supplementary materials?

Return the CORRECTED final JSON only:
{{"disease": "disease name", "taxa_enriched": ["taxon1", "taxon2"], "taxa_depleted": ["taxon3", "taxon4"]}}

Paper text:
{text}

Corrected JSON:"""
'''

In [ ]:
#2 prompt break down approach not work 
'''def smart_truncate(text):
    for marker in ["References\n", "REFERENCES\n", "Bibliography\n"]:
        idx = text.rfind(marker)
        if idx > 0:
            return text[:idx]
    return text

def parse_output(raw):
    clean = raw
    if "</think>" in clean:
        clean = clean.split("</think>")[-1]
    clean = clean.strip()
    if "```json" in clean:
        clean = clean.split("```json")[1].split("```")[0]
    elif "```" in clean:
        clean = clean.split("```")[1].split("```")[0]
    return json.loads(clean.strip())

results = []

for i, paper in enumerate(papers):
    print(f"[{i+1}/{len(papers)}] {paper['title'][:70]}...")
    text = smart_truncate(paper["text"])
    
    # Call 1: simple extraction
    start = time.time()
    out1 = llm.create_chat_completion(
        messages=[{"role": "user", "content": PROMPT_CALL1.format(text=text)}],
        temperature=0, max_tokens=2048,
    )
    try:
        parsed1 = parse_output(out1["choices"][0]["message"]["content"])
    except:
        parsed1 = {"disease": paper["disease"], "taxa_enriched": [], "taxa_depleted": []}
    
    disease = parsed1.get("disease", "unknown")
    print(f"  Call 1: {disease}, {len(parsed1.get('taxa_enriched',[]))} enriched, {len(parsed1.get('taxa_depleted',[]))} depleted")
    
    # Call 2: self-refine
    out2 = llm.create_chat_completion(
        messages=[{"role": "user", "content": PROMPT_CALL2.format(
            disease=disease,
            first_pass_json=json.dumps(parsed1),
            text=text
        )}],
        temperature=0, max_tokens=2048,
    )
    try:
        parsed2 = parse_output(out2["choices"][0]["message"]["content"])
    except:
        parsed2 = parsed1  # fall back to Call 1 if Call 2 fails
    
    elapsed = time.time() - start
    
    print(f"  Call 2: enriched={parsed2.get('taxa_enriched', [])}")
    print(f"          depleted={parsed2.get('taxa_depleted', [])}")
    print(f"  {elapsed:.1f}s")
    
    results.append({
        "title": paper["title"],
        "disease": paper["disease"],
        "in_gold_standard": paper["in_gold_standard"],
        "expected_enriched": paper.get("taxa_enriched", ""),
        "expected_depleted": paper.get("taxa_depleted", ""),
        "predicted_enriched": ", ".join(parsed2.get("taxa_enriched", [])),
        "predicted_depleted": ", ".join(parsed2.get("taxa_depleted", [])),
        "predicted_disease": disease,
        "call1_enriched": ", ".join(parsed1.get("taxa_enriched", [])),
        "call1_depleted": ", ".join(parsed1.get("taxa_depleted", [])),
        "time_seconds": round(elapsed, 2),
        "parse_error": False,
    })

print(f"\nDone. {len(results)} papers.")'''

[1/15] Intestinal flora induces depression by mediating the dysregulation of ...
  Call 1: Stroke, 0 enriched, 0 depleted
  Call 2: enriched=[]
          depleted=[]
  93.4s
[2/15] Gut microbes exacerbate systemic inflammation and behavior disorders i...
  Call 1: Other, 0 enriched, 0 depleted
  Call 2: enriched=[]
          depleted=[]
  56.4s
[3/15] Alterations in gut microbiota and metabolomic profiles in acute stroke...
  Call 1: Stroke, 0 enriched, 0 depleted
  Call 2: enriched=[]
          depleted=[]
  95.9s
[4/15] Gut microbiome dysbiosis across early Parkinson's disease, REM sleep b...
  Call 1: Parkinson’s disease (PD), 0 enriched, 0 depleted
  Call 2: enriched=[]
          depleted=[]
  99.4s
[5/15] The gut microbiota in multiple sclerosis varies with disease activity....


In [11]:
run_name = "decomposed_v3"  # change this each run
tag = f"qwopus_{run_name}"

json_path = f"benchmark_results/{tag}.json"
with open(json_path, "w") as f:
    json.dump(results, f, indent=2)
csv_path = f"benchmark_results/{tag}.csv"
with open(csv_path, "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=results[0].keys())
    w.writeheader()
    w.writerows(results)

print(f"Saved: {json_path}")
print(f"Saved: {csv_path}")

Saved: benchmark_results/qwopus_decomposed_v3.json
Saved: benchmark_results/qwopus_decomposed_v3.csv


In [12]:
#if want to run eval on custom file
import csv
import re
with open("benchmark_results/qwopus_v3_decomposed_v1.csv", "r") as f:
    results = list(csv.DictReader(f))

In [14]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

def parse_taxa(val):
    if not val or str(val) == "NaN" or str(val) == "nan":
        return []
    taxa = re.split(r'[,;]', str(val))
    cleaned = []
    for t in taxa:
        t = re.sub(r'\(.*?\)', '', t).strip().lower()
        t = re.sub(r'p\s*[<>=]\s*[\d.]+', '', t).strip()
        t = t.strip('.) ')
        if t and t != "nan" and len(t) > 2:
            cleaned.append(t)
    return cleaned

def match_taxa(predicted, expected):
    if not predicted and not expected:
        return [], []
    if not predicted:
        return [], expected
    if not expected:
        return [(p, "N/A (no expected)", 0.0) for p in predicted], []
    all_names = predicted + expected
    tfidf = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4)).fit_transform(all_names)
    sim = cosine_similarity(tfidf[:len(predicted)], tfidf[len(predicted):])
    matches = []
    matched_expected_idx = set()
    for i, pred in enumerate(predicted):
        best_j = sim[i].argmax()
        score = round(float(sim[i][best_j]), 3)
        matches.append((pred, expected[best_j], score))
        if score >= 0.5:
            matched_expected_idx.add(best_j)
    missed = [expected[j] for j in range(len(expected)) if j not in matched_expected_idx]
    return matches, missed

total_tp = total_fp = total_fn = 0

for r in results:
    print(f"\n{'='*60}")
    print(f"{r['title'][:70]}")
    print(f"Disease: {r['disease']} | Gold: {r['in_gold_standard']}")
    if 'taxa_from_call1' in r:
        print(f"  Call 1 taxa: {r['taxa_from_call1'][:100]}...")
    pred_enr = parse_taxa(r["predicted_enriched"])
    pred_dep = parse_taxa(r["predicted_depleted"])
    exp_enr = parse_taxa(r["expected_enriched"])
    exp_dep = parse_taxa(r["expected_depleted"])
    enr_matches, enr_missed = match_taxa(pred_enr, exp_enr)
    dep_matches, dep_missed = match_taxa(pred_dep, exp_dep)
    if enr_matches:
        print("  ENRICHED:")
        for pred, exp, score in enr_matches:
            flag = "✅" if score >= 0.5 else "❌"
            print(f"    {flag} {pred} → {exp}  sim={score}")
    if enr_missed:
        print("  MISSED ENRICHED:")
        for m in enr_missed:
            print(f"    ⚠️  {m}")
    if dep_matches:
        print("  DEPLETED:")
        for pred, exp, score in dep_matches:
            flag = "✅" if score >= 0.5 else "❌"
            print(f"    {flag} {pred} → {exp}  sim={score}")
    if dep_missed:
        print("  MISSED DEPLETED:")
        for m in dep_missed:
            print(f"    ⚠️  {m}")
    if not enr_matches and not dep_matches and not enr_missed and not dep_missed:
        print("  (no predictions, no expected)")
    tp = sum(1 for _, _, s in enr_matches + dep_matches if s >= 0.5)
    fp = sum(1 for _, _, s in enr_matches + dep_matches if s < 0.5)
    fn = len(enr_missed) + len(dep_missed)
    print(f"  → TP={tp}  FP={fp}  FN={fn}")
    total_tp += tp
    total_fp += fp
    total_fn += fn

prec = total_tp / (total_tp + total_fp) if (total_tp + total_fp) else 0
rec = total_tp / (total_tp + total_fn) if (total_tp + total_fn) else 0
f1 = 2 * prec * rec / (prec + rec) if (prec + rec) else 0
print(f"\n{'='*60}")
print(f"TOTAL: TP={total_tp}  FP={total_fp}  FN={total_fn}")
print(f"Precision: {prec:.3f}  Recall: {rec:.3f}  F1: {f1:.3f}")


Intestinal flora induces depression by mediating the dysregulation of 
Disease: Stroke | Gold: No
  MISSED ENRICHED:
    ⚠️  g_erysipelotrichaceae ucg-003
  → TP=0  FP=0  FN=1

Gut microbes exacerbate systemic inflammation and behavior disorders i
Disease: Other | Gold: Yes
  ENRICHED:
    ✅ eubacterium siraeum → eubacterium siraeum  sim=1.0
    ✅ akkermansia muciniphila → akkermansia muciniphila  sim=1.0
    ✅ fusobacterium varium → fusobacterium varium  sim=1.0
    ✅ megasphaera elsdenii → megasphaera elsdenii  sim=1.0
    ✅ clostridium aldenense → clostridium aldenense  sim=1.0
    ✅ bacteroides sp. om05-12 → bacteroides sp. om05-12  sim=1.0
    ✅ streptococcus sp. a12 → streptococcus sp. a12  sim=1.0
  MISSED ENRICHED:
    ⚠️  porphyromonas
    ⚠️  akkermansia
  DEPLETED:
    ✅ roseburia faecis → roseburia faecis  sim=1.0
    ✅ eubacterium eligens → eubacterium eligens  sim=1.0
    ✅ sutterella parvirubra → sutterella parvirubra  sim=1.0
  MISSED DEPLETED:
    ⚠️  sutterella
  → T

In [15]:
import os

paper_dir = "benchmark_results"  # adjust

for r in results:
    exp = parse_taxa(r["expected_enriched"]) + parse_taxa(r["expected_depleted"])
    pred = parse_taxa(r["predicted_enriched"]) + parse_taxa(r["predicted_depleted"])
    
    if exp and not pred:
        title = r['title'][:70]
        # try to find matching file
        for fname in os.listdir(paper_dir):
            fpath = os.path.join(paper_dir, fname)
            size = os.path.getsize(fpath)
            print(f"  {title}")
            print(f"    File: {fname} | Size: {size/1024:.1f} KB")
            break  #

  Intestinal flora induces depression by mediating the dysregulation of 
    File: qwopus_cot_updatedprompt_20260329_050629.csv | Size: 7.3 KB
  Gut Microbial Ecosystem in Parkinson Disease: New Clinicobiological In
    File: qwopus_cot_updatedprompt_20260329_050629.csv | Size: 7.3 KB
